# Notebook 04.1 — Category Clustering with BGE + BERTopic

This notebook replaces the nomic-embed + MiniBatchKMeans pipeline from Notebook 04 with a **BGE-large-en-v1.5 embedding + BERTopic clustering** approach. The goal is to discover product-category clusters that are **not driven by sentiment**, using density-based clustering (HDBSCAN) with a KMeans fallback.

**Inputs**: Arrow dataset from N01, predictions_distilbert.csv from N02  
**Outputs**: cluster_assignments.json, cluster_profiles_bge.json, umap_coordinates.json, embeddings_bge.npz


In [ ]:
# ── 0.1  Detect execution environment & mount Drive ────────────────────────
# I use try/except ImportError to detect Colab — this is robust against version changes.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running in Colab: {IN_COLAB}")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted at /content/drive")
else:
    print("Skipping Drive mount — running locally.")


In [ ]:
# ── 0.2  Install required packages ─────────────────────────────────────────
# BERTopic pulls UMAP and HDBSCAN, but I install them explicitly for version control.
if IN_COLAB:
    import subprocess
    subprocess.run([
        "pip", "install", "-q", "--upgrade",
        "sentence-transformers",   # BGE embedding model
        "bertopic",                # Topic modelling + clustering
        "hdbscan",                 # Density-based clustering
        "umap-learn",              # Dimensionality reduction
        "datasets",                # HuggingFace Datasets (Arrow)
        "pyarrow",                 # Arrow backend
    ], check=True)
    print("All packages installed.")
else:
    print("Skipping pip install — manage dependencies locally.")


In [ ]:
# ── 0.3  Standard library and third-party imports ──────────────────────────
# ALL imports live here. No scattered imports in processing cells (AGENTS.md §5).
import os
import json
import html
import math
import time
import random
import re
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from datasets import load_from_disk, concatenate_datasets
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from hdbscan import HDBSCAN
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer
from scipy.stats import entropy
import umap
from tqdm.auto import tqdm

print("All imports successful.")


In [ ]:
# ── 0.4  Set random seeds for full reproducibility ─────────────────────────
# Fixing seeds in Python, NumPy, and PyTorch ensures deterministic results.
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"Random seeds fixed to {RANDOM_SEED}.")


In [ ]:
# ── 0.5  Detect and configure compute device ───────────────────────────────
# BGE-large runs efficiently on GPU; sentence-transformers auto-uses it.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute device: {DEVICE}")
if DEVICE.type == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU model : {gpu_name}")
    print(f"GPU memory: {total_mem:.1f} GB")
else:
    print("No GPU detected — embeddings will run on CPU.")


In [ ]:
# ── 0.6  Configure display and plotting defaults ───────────────────────────
# Palette from design-visual.md §Rules. Grid, edgecolor, and dpi are fixed globally.
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 20)

CYAN = "#0891B2"
NARANJA = "#EA580C"
RED = "#DC2626"
SLATE = "#334155"
NEO_SUCCESS = "#10B981"
SENTIMENT_COLORS = ["#DC2626", "#94A3B8", "#10B981"]   # Neg, Neu, Pos
CLUSTER_COLORS = ["#0891B2", "#7C3AED", "#D97706", "#0D9488", "#4F46E5", "#BE185D"]
GRID_COLOR = "#94A3B8"

plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["grid.color"] = GRID_COLOR
plt.rcParams["grid.linestyle"] = "--"
plt.rcParams["grid.alpha"] = "0.4"

print("Display defaults configured.")


In [ ]:
# ── 0.7  Define all project paths ──────────────────────────────────────────
# Centralised paths so changing them once propagates everywhere.
if IN_COLAB:
    BASE_DIR = "/content/drive/MyDrive/nlp-project/business-case-01"
else:
    BASE_DIR = os.path.expanduser("~/+Dev/nlp-businesscase")

DATASET_DIR = os.path.join(BASE_DIR, "data", "dataset")
OUTPUT_DIR  = os.path.join(BASE_DIR, "data")
PLOTS_DIR   = os.path.join(BASE_DIR, "data", "plots")
MODELS_DIR  = os.path.join(BASE_DIR, "data", "models")

os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"BASE_DIR    : {BASE_DIR}")
print(f"DATASET_DIR : {DATASET_DIR}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")
print(f"PLOTS_DIR   : {PLOTS_DIR}")
print(f"MODELS_DIR  : {MODELS_DIR}")


## Section 1 — Data Loading

I load the Arrow dataset and merge with DistilBERT sentiment predictions.

In [ ]:
# ── 1.0  Validate upstream artifacts ───────────────────────────────────────
# Guard-check before any heavy computation. Catching a missing file here
# saves a cryptic traceback 20 cells later.
for split in ["train", "validation", "test"]:
    split_path = os.path.join(DATASET_DIR, split)
    assert os.path.exists(split_path), f"N01 output missing: {split_path}"

PRED_PATH = os.path.join(OUTPUT_DIR, "predictions_distilbert.csv")
assert os.path.exists(PRED_PATH), (
    f"predictions_distilbert.csv not found at {PRED_PATH}. Run N02 first."
)

print("✅ Arrow splits and predictions_distilbert.csv verified.")


In [ ]:
# ── 1.1  Load Arrow dataset — all splits ───────────────────────────────────
# I concatenate train+validation+test to maximise clustering signal (~214K reviews).
dataset = load_from_disk(DATASET_DIR)

df_all = pd.concat(
    [
        dataset["train"].to_pandas(),
        dataset["validation"].to_pandas(),
        dataset["test"].to_pandas(),
    ],
    ignore_index=True,
)

print(f"Full dataset shape: {df_all.shape}")
print(f"Columns: {list(df_all.columns)}")
print(f"\nLabel distribution:")
print(df_all["label"].value_counts().sort_index())


In [ ]:
# ── 1.2  Load sentiment predictions from N02 ───────────────────────────────
# DistilBERT (F1=0.77) is the primary prediction source.
df_preds = pd.read_csv(PRED_PATH, encoding="utf-8")
print(f"Loaded predictions: {len(df_preds):,} rows")
print(f"Columns: {list(df_preds.columns)}")


In [ ]:
# ── 1.3  Merge data with sentiment predictions ─────────────────────────────
# Predictions only cover the test split (~15%), so I use TRUE labels (100% coverage)
# as the primary sentiment signal. Predicted labels enrich the export JSON.
df_merged = df_all.merge(
    df_preds[["text", "predicted_label", "confidence"]],
    on="text",
    how="left",
)

# Primary sentiment column — true labels, 100% coverage
unique_labels = sorted(df_merged["label"].unique())
label_names = {lbl: name for lbl, name in zip(unique_labels, ["Negative", "Neutral", "Positive"])}
df_merged["sentiment_name"] = df_merged["label"].map(label_names)

# Guard: stop if merge drops >1% rows (indicates key mismatch)
drop_pct = 100 * (len(df_all) - len(df_merged)) / len(df_all)
assert drop_pct <= 1.0, f"Merge dropped {drop_pct:.1f}% rows — investigate."

print(f"Merged shape: {df_merged.shape}")
print(f"Missing values per column:")
print(df_merged.isnull().sum())


In [ ]:
# ── 1.4  Sampling strategy for large datasets ──────────────────────────────
# With ~214K reviews, no sampling is usually needed. The cap at 220K is a safety valve.
MAX_SAMPLES = 220_000

if len(df_merged) > MAX_SAMPLES:
    print(f"Dataset has {len(df_merged):,} reviews — sampling {MAX_SAMPLES:,}...")
    rng = np.random.default_rng(RANDOM_SEED)
    sample_indices = []
    for label_val in sorted(df_merged["label"].unique()):
        group_mask = df_merged["label"] == label_val
        group_idx = df_merged.index[group_mask]
        n_for_group = max(1, int(MAX_SAMPLES * len(group_idx) / len(df_merged)))
        n_for_group = min(len(group_idx), n_for_group)
        chosen = rng.choice(group_idx, size=n_for_group, replace=False)
        sample_indices.extend(chosen.tolist())
    df_merged = df_merged.loc[sample_indices].reset_index(drop=True)
    print(f"  → Sampled down to {len(df_merged):,} reviews.")
else:
    print(f"Dataset has {len(df_merged):,} reviews — no sampling needed.")

print("\nClass distribution:")
for lbl, count in df_merged["label"].value_counts().sort_index().items():
    pct = 100 * count / len(df_merged)
    print(f"  {label_names[lbl]:10s} ({lbl}): {count:>8,}  ({pct:5.1f}%)")


---
## Section 1.5 — Emotional Lexicon Filtering

Before embedding, I remove evaluative/emotional vocabulary that dominates the
embedding space. Sentence embeddings (BGE, MiniLM, nomic) are contrastively trained —
words like "great", "terrible", "love", "hate" carry strong similarity signals.
If left in, the clustering groups reviews by **sentiment tone** rather than
**product category** — the exact problem we diagnosed in N04.

### Strategy

| Tier | Type | Size | Example |
|------|------|------|---------|
| 1 | Pure emotion | 40 words | amazing, terrible, love, hate, worst |
| 2 | Value judgment | ~110 words/phrases | good product, overpriced, highly recommend |

Regex-based word-boundary removal before the embedding step. The filtered text is
saved as `reviews_filtered.csv` — a checkpoint so downstream cells can resume from
disk without re-running this filter.

**Output**: `data/reviews_filtered.csv` (text + text_filtered + label + rating + category)

In [ ]:
# ── 1.5  Emotional lexicon filter ────────────────────────────────────────
# Remove evaluative vocabulary before embedding so the clustering captures
# product-category signal, not sentiment tone.
#
# Expected output: df_merged["text_filtered"] column + reviews_filtered.csv checkpoint.

# Tier 1 — Pure emotion (never descriptive in Amazon reviews):
STOPWORDS_TIER1 = [
    "amazing", "awesome", "awful", "beautiful", "best", "brilliant",
    "crap", "delighted", "disappointed", "dreadful", "excellent",
    "fantastic", "favorite", "garbage", "glad", "good product", "great",
    "happy", "hate", "horrible", "incredible", "love", "loved", "lousy",
    "magnificent", "mediocre", "nice", "outstanding", "perfect", "pleased",
    "rubbish", "satisfied", "superb", "terrible", "terrific", "thrilled",
    "useless", "waste", "wonderful", "worst", "worth",
]

# Tier 2 — Value judgments (rarely descriptive in reviews):
STOPWORDS_TIER2 = [
    "boring", "decent", "disappointing", "enjoyable", "excellent product",
    "fair", "fantastic product", "fine product", "fun", "good quality",
    "good value", "highly recommend", "impressive", "junk", "life changing",
    "love it", "must buy", "not bad", "not recommended", "not worth",
    "ok product", "overpriced", "poor", "poor quality", "regret",
    "return it", "rip off", "so good", "terrible product", "very good",
    "very happy", "very pleased", "waste of money", "would not buy",
    "zero stars",
]

ALL_STOPWORDS = STOPWORDS_TIER1 + STOPWORDS_TIER2
print(f"Emotional stopwords loaded: {len(STOPWORDS_TIER1)} (Tier 1) + {len(STOPWORDS_TIER2)} (Tier 2) = {len(ALL_STOPWORDS)} total")


def filter_emotional_lexicon(text: str) -> str:
    """Remove evaluative/emotional words from review text.

    Uses case-insensitive regex with word-boundary matching so we don't
    accidentally remove substrings (e.g. "worth" in "worthwhile").
    Multi-word phrases (e.g. "good product") match as exact phrases.
    Preserves original spacing and punctuation.
    """

    filtered = text
    # Sort by length descending — longer phrases match first to avoid
    # partial removal (e.g. "waste of money" before "waste").
    sorted_words = sorted(ALL_STOPWORDS, key=len, reverse=True)

    for word in sorted_words:
        pattern = r'\b' + re.escape(word) + r'\b'
        filtered = re.sub(pattern, '', filtered, flags=re.IGNORECASE)

    # Collapse multiple spaces into one, trim edges
    filtered = re.sub(r'\s+', ' ', filtered).strip()
    return filtered


# ── Apply filter ──────────────────────────────────────────────────────────
# Guard: df_merged must exist from Section 1.
if "df_merged" not in dir():
    raise NameError("Run Section 1 first (cell-load-arrow through cell-sample).")

print("\nFiltering emotional lexicon from review texts...")
start = time.perf_counter()
df_merged["text_filtered"] = df_merged["text"].apply(filter_emotional_lexicon)
elapsed = time.perf_counter() - start

# ── Verification ───────────────────────────────────────────────────────────
n_total = len(df_merged)
n_changed = (df_merged["text"] != df_merged["text_filtered"]).sum()
pct_changed = 100 * n_changed / n_total

print(f"\n  Rows processed : {n_total:,}")
print(f"  Rows modified  : {n_changed:,} ({pct_changed:.1f}%)")
print(f"  Time           : {elapsed:.1f}s")

# Before/After examples — random sample (not .head(3), AGENTS.md trap #4)
print(f"\n  Before/After examples (random sample):")
sample = df_merged.sample(n=3, random_state=RANDOM_SEED)
for i, (_, row) in enumerate(sample.iterrows(), 1):
    orig = str(row["text"])
    filt = str(row["text_filtered"])
    print(f"\n  [{i}] Original : {orig[:200]}{'…' if len(orig) > 200 else ''}")
    print(f"      Filtered: {filt[:200]}{'…' if len(filt) > 200 else ''}")

# ── Save checkpoint ────────────────────────────────────────────────────────
FILTERED_CSV = os.path.join(OUTPUT_DIR, "reviews_filtered.csv")
df_merged[["text", "text_filtered", "label", "rating", "category"]].to_csv(
    FILTERED_CSV, index=False
)
file_size_mb = os.path.getsize(FILTERED_CSV) / 1e6
print(f"\n  ✓ Filtered texts saved → {FILTERED_CSV}")
print(f"    Columns: text, text_filtered, label, rating, category")
print(f"    Rows   : {len(df_merged):,}")
print(f"    Size   : {file_size_mb:.1f} MB")
print(f"\n  → Next: Section 2 will embed text_filtered (not raw text).")


## Section 2 — BGE-large Embeddings

I convert each review into a 1024-dimensional vector using `BAAI/bge-large-en-v1.5`. Contrastive training → better topic/tone separation.

In [ ]:
# ── 2.1  Load BGE-large-en-v1.5 embedding model ────────────────────────────
# BGE-large-en-v1.5: 1024-dim, 335M params. Better topic/tone separation than nomic-embed.
print("Loading BAAI/bge-large-en-v1.5 ...")
embedding_model = SentenceTransformer("BAAI/bge-large-en-v1.5", device=str(DEVICE))
embedding_model.max_seq_length = 256   # Covers ~99% of Amazon reviews (median ~52 tokens)

print(f"Model loaded: BAAI/bge-large-en-v1.5")
print(f"  Embedding dim : {embedding_model.get_sentence_embedding_dimension()}")
print(f"  Max seq length: {embedding_model.max_seq_length}")
print(f"  Device        : {DEVICE}")


In [ ]:
# ── 2.2  Encode filtered review texts ───────────────────────────────────────────
# Guard: ensure DataFrame exists before encoding.
if "df_merged" not in dir():
    raise NameError("Run Section 1 first (cell-load-arrow through cell-sample).")

# html.unescape() BEFORE encoding — AGENTS.md trap #1: residual &amp; / &quot; produce noise vectors.
review_texts = [html.unescape(t) for t in df_merged["text_filtered"].tolist()]

print(f"Encoding {len(review_texts):,} reviews (batch_size=32, normalized)...")
start = time.perf_counter()
embeddings = embedding_model.encode(
    review_texts,
    batch_size=32,                 # T4 VRAM safe: 32×1024×4B ≈ 128MB per batch
    show_progress_bar=True,
    normalize_embeddings=True,
)
elapsed = time.perf_counter() - start

print(f"\nEmbeddings shape : {embeddings.shape}")
assert embeddings.shape == (len(df_merged), 1024), f"Dimension mismatch: expected ({len(df_merged)}, 1024), got {embeddings.shape}"
print(f"  dtype          : {embeddings.dtype}")
print(f"  Memory usage   : {embeddings.nbytes / 1e9:.2f} GB")
print(f"  Time           : {elapsed/60:.1f} min")

# Sanity check: norms should be ~1.0 after normalisation
norms = np.linalg.norm(embeddings, axis=1)
print(f"  Mean L2 norm   : {norms.mean():.4f} (expected ≈ 1.0)")


In [ ]:
# ── 2.3  Save embeddings to disk ───────────────────────────────────────────
npz_path = os.path.join(MODELS_DIR, "embeddings_bge.npz")
np.savez_compressed(npz_path, embeddings=embeddings)

file_size_mb = os.path.getsize(npz_path) / 1e6
print(f"Embeddings saved → {npz_path}")
print(f"  File size: {file_size_mb:.1f} MB")


## Section 3 — BERTopic Clustering

HDBSCAN with retry ladder (4–6 clusters). KMeans fallback if out of range.

In [ ]:
# ── 2.4  Verify embedding quality — cosine similarity check ────────────────
# I use .sample(n=5) (not .head(3)) for non-trivial verification — AGENTS.md trap #4.
sampled = df_merged.sample(n=5, random_state=RANDOM_SEED).reset_index(drop=True)
sample_emb = embeddings[sampled.index]

# Cosine similarity on normalised vectors = dot product
sim_matrix = np.dot(sample_emb, sample_emb.T)

print("Cosine similarity matrix (sampled reviews):")
print(sim_matrix.round(4))
print()

same_class_scores = []
diff_class_scores = []
for i in range(len(sampled)):
    for j in range(i + 1, len(sampled)):
        score = sim_matrix[i, j]
        if sampled.loc[i, "label"] == sampled.loc[j, "label"]:
            same_class_scores.append(score)
        else:
            diff_class_scores.append(score)

if same_class_scores and diff_class_scores:
    same_mean = np.mean(same_class_scores)
    diff_mean = np.mean(diff_class_scores)
    margin = same_mean - diff_mean
    print(f"Same-class pairs     : {len(same_class_scores)}  |  mean similarity: {same_mean:.4f}")
    print(f"Different-class pairs: {len(diff_class_scores)}  |  mean similarity: {diff_mean:.4f}")
    print(f"Margin (same - diff) : {margin:.4f}")
    if margin >= 0.05:
        print("✅ Embeddings pass sanity check — similar-class reviews are closer.")
    else:
        print("⚠️  Margin < 0.05 — embeddings may not separate classes well.")
else:
    print("⚠️  Not enough same-class or different-class pairs in sample.")


In [ ]:
# ── 3.1  Configure BERTopic with UMAP + HDBSCAN ────────────────────────────
# Primary config targets 4–6 clusters on ~220K reviews.
umap_model = umap.UMAP(
    n_neighbors=15,
    n_components=5,
    metric="cosine",
    random_state=RANDOM_SEED,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=500,
    metric="euclidean",
    min_samples=10,
    prediction_data=True,
)

vectorizer_model = CountVectorizer(stop_words="english")

print("BERTopic configuration:")
print(f"  UMAP    : n_neighbors=15, n_components=5, metric=cosine, seed={RANDOM_SEED}")
print(f"  HDBSCAN : min_cluster_size=500, metric=euclidean, min_samples=10")
print(f"  Vectorizer: CountVectorizer(stop_words='english')")


In [ ]:
# ── 3.2  Fit BERTopic ──────────────────────────────────────────────────────
# Guard: embeddings must exist from Section 2.
if "embeddings" not in dir():
    raise NameError("Run Section 2 first to generate embeddings.")

docs = df_merged["text_filtered"].tolist()

print("Fitting BERTopic (this may take 5–10 minutes)...")
start = time.perf_counter()
topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    verbose=True,
)
topics, probs = topic_model.fit_transform(docs, embeddings)
elapsed = time.perf_counter() - start

print(f"\nFitting completed in {elapsed/60:.1f} minutes.")
print(f"Number of topics (including noise -1): {len(topic_model.get_topic_info())}")


In [ ]:
# ── 3.3  Validate cluster count and retry if needed ────────────────────────
# Dynamically derive k — never hardcode (AGENTS.md §Hardcoding traps).
labels = np.array(topics)
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = (labels == -1).sum()

print(f"Initial HDBSCAN result:")
print(f"  n_clusters : {n_clusters}")
print(f"  n_noise    : {n_noise} ({100*n_noise/len(labels):.2f}%)")

algorithm_used = "HDBSCAN"
final_min_cluster_size = 500

# Retry ladder: <4 → reduce granularity; >6 → increase granularity
if n_clusters < 4:
    for mcs in [300, 200, 100]:
        print(f"\nRetrying with min_cluster_size={mcs}...")
        hdbscan_retry = HDBSCAN(
            min_cluster_size=mcs, metric="euclidean", min_samples=10, prediction_data=True
        )
        topic_model = BERTopic(
            umap_model=umap_model,
            hdbscan_model=hdbscan_retry,
            vectorizer_model=vectorizer_model,
            verbose=True,
        )
        topics, probs = topic_model.fit_transform(docs, embeddings)
        labels = np.array(topics)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        print(f"  → n_clusters={n_clusters}")
        if 4 <= n_clusters <= 6:
            final_min_cluster_size = mcs
            break

elif n_clusters > 6:
    for mcs in [800, 1000]:
        print(f"\nRetrying with min_cluster_size={mcs}...")
        hdbscan_retry = HDBSCAN(
            min_cluster_size=mcs, metric="euclidean", min_samples=10, prediction_data=True
        )
        topic_model = BERTopic(
            umap_model=umap_model,
            hdbscan_model=hdbscan_retry,
            vectorizer_model=vectorizer_model,
            verbose=True,
        )
        topics, probs = topic_model.fit_transform(docs, embeddings)
        labels = np.array(topics)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        print(f"  → n_clusters={n_clusters}")
        if 4 <= n_clusters <= 6:
            final_min_cluster_size = mcs
            break

# Fallback to KMeans if still out of range after 3 retries
if not (4 <= n_clusters <= 6):
    print(f"\nFallback to KMeans(n_clusters=6, random_state={RANDOM_SEED})...")
    kmeans_model = KMeans(n_clusters=6, random_state=RANDOM_SEED, n_init=10)
    topic_model = BERTopic(
        umap_model=umap_model,
        hdbscan_model=kmeans_model,
        vectorizer_model=vectorizer_model,
        verbose=True,
    )
    topics, probs = topic_model.fit_transform(docs, embeddings)
    labels = np.array(topics)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    algorithm_used = "KMeans(n=6)"
    final_min_cluster_size = None
    print(f"  → n_clusters={n_clusters}")

# Persist cluster labels to DataFrame for downstream cells
df_merged["cluster"] = labels
n_noise = (labels == -1).sum()

print(f"\n{'='*50}")
print(f"Final clustering result:")
print(f"  Algorithm : {algorithm_used}")
print(f"  n_clusters: {n_clusters}")
print(f"  n_noise   : {n_noise} ({100*n_noise/len(labels):.2f}%)")
if final_min_cluster_size:
    print(f"  min_cluster_size used: {final_min_cluster_size}")


In [ ]:
# ── 3.4  Cluster size distribution ─────────────────────────────────────────
# Bar chart using CLUSTER_COLORS from design-visual.md. Noise (-1) shown separately.
cluster_counts = df_merged[df_merged["cluster"] != -1]["cluster"].value_counts().sort_index()

print("Cluster size distribution (excluding noise):")
print(f"  {'Cluster':>8s}  |  {'Count':>8s}  |  {'%':>6s}")
print(f"  {'-'*32}")
for cluster_id in sorted(cluster_counts.index):
    count = cluster_counts[cluster_id]
    pct = 100 * count / len(df_merged)
    print(f"  {cluster_id:8.0f}  |  {count:>8,}  |  {pct:>5.1f}%")

if -1 in df_merged["cluster"].values:
    noise_count = (df_merged["cluster"] == -1).sum()
    noise_pct = 100 * noise_count / len(df_merged)
    print(f"  {'Noise (-1)':>8s}  |  {noise_count:>8,}  |  {noise_pct:>5.1f}%")

# Bar chart ──
fig, ax = plt.subplots(figsize=(8, 5))
n_clusters_found = len(cluster_counts)
colors = CLUSTER_COLORS[:n_clusters_found]
bars = ax.bar(cluster_counts.index, cluster_counts.values, color=colors, edgecolor="white", linewidth=0.8)

ax.set_title(f"Cluster Size Distribution (k={n_clusters_found})", fontsize=13, fontweight="bold")
ax.set_xlabel("Cluster ID", fontsize=11)
ax.set_ylabel("Number of Reviews", fontsize=11)

max_height = cluster_counts.max()
ax.set_ylim(0, max_height * 1.2)

for bar, (cluster_id, count) in zip(bars, cluster_counts.items()):
    pct = 100 * count / len(df_merged)
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max_height * 0.02,
        f"{count:,}\n({pct:.1f}%)",
        ha="center", va="bottom", fontsize=9,
    )

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, "nb04_1_cluster_sizes.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"\n✅ Plot saved → {plot_path}")
plt.show()

# Top c-TF-IDF terms per cluster
print("\nTop c-TF-IDF terms per cluster:")
for topic_id in sorted(cluster_counts.index):
    topic_words = topic_model.get_topic(topic_id)
    if topic_words:
        words = [w for w, _ in topic_words[:5]]
        print(f"  Cluster {int(topic_id)}: {', '.join(words)}")
    else:
        print(f"  Cluster {int(topic_id)}: (no terms)")


# Verify c-TF-IDF coherence
for topic_id in sorted(topic_model.topics_):
    if topic_id == -1:
        continue
    top_words = topic_model.get_topic(topic_id)
    if top_words:
        avg_score = np.mean([score for _, score in top_words[:10]])
        status = "✅" if avg_score > 0.3 else "⚠️ "
        print(f"  {status} Topic {topic_id}: avg c-TF-IDF score = {avg_score:.4f}")

## Section 4 — UMAP Visualization

2D projection of BERTopic's internal embeddings, colored by cluster and sentiment.

In [ ]:
# ── 4.1  Stratified sample for UMAP visualization ──────────────────────────
# Guard: cluster labels must exist from Section 3.
if "df_merged" not in dir() or "cluster" not in df_merged.columns:
    raise NameError("Run Section 3 first to generate cluster labels.")

SAMPLE_SIZE = 5000

rng = np.random.default_rng(RANDOM_SEED)
sample_indices = []
for cid in sorted(df_merged["cluster"].unique()):
    mask = df_merged["cluster"] == cid
    idx = df_merged.index[mask]
    n_sample = max(1, int(SAMPLE_SIZE * len(idx) / len(df_merged)))
    n_sample = min(len(idx), n_sample)
    chosen = rng.choice(idx, size=n_sample, replace=False)
    sample_indices.extend(chosen.tolist())

df_sample = df_merged.loc[sample_indices].copy()
emb_sample = embeddings[sample_indices]

print(f"Stratified sample: {len(df_sample):,} reviews")
print(f"Cluster distribution in sample:")
print(df_sample["cluster"].value_counts().sort_index())


In [ ]:
# ── 4.2  UMAP scatter coloured by cluster ──────────────────────────────────
# Re-fit UMAP to 2D for visualization (BERTopic's internal UMAP is 5D).
umap_2d = umap.UMAP(
    n_neighbors=15,
    n_components=2,
    metric="cosine",
    random_state=RANDOM_SEED,
)
umap_coords = umap_2d.fit_transform(emb_sample)

df_sample["umap_x"] = umap_coords[:, 0]
df_sample["umap_y"] = umap_coords[:, 1]

fig, ax = plt.subplots(figsize=(10, 8))
cluster_ids = sorted([c for c in df_sample["cluster"].unique() if c != -1])

for i, cid in enumerate(cluster_ids):
    mask = df_sample["cluster"] == cid
    ax.scatter(
        df_sample.loc[mask, "umap_x"],
        df_sample.loc[mask, "umap_y"],
        c=[CLUSTER_COLORS[i % len(CLUSTER_COLORS)]],
        label=f"Cluster {int(cid)}",
        s=8,
        alpha=0.6,
        edgecolors="none",
    )

ax.set_title("UMAP Projection — Coloured by Cluster", fontsize=14, fontweight="bold")
ax.set_xlabel("UMAP Dimension 1", fontsize=11)
ax.set_ylabel("UMAP Dimension 2", fontsize=11)
ax.legend(markerscale=3, fontsize=10, title="Cluster", title_fontsize=11, loc="upper right")
ax.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, "nb04_1_umap_clusters.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"✅ Plot saved → {plot_path}")
plt.show()


In [ ]:
# ── 4.3  UMAP scatter coloured by sentiment ────────────────────────────────
# This validates that clusters are NOT sentiment-segregated.
fig, ax = plt.subplots(figsize=(10, 8))

sentiment_labels_list = sorted(df_sample["label"].unique())
for i, lbl in enumerate(sentiment_labels_list):
    mask = df_sample["label"] == lbl
    ax.scatter(
        df_sample.loc[mask, "umap_x"],
        df_sample.loc[mask, "umap_y"],
        c=[SENTIMENT_COLORS[i % len(SENTIMENT_COLORS)]],
        label=label_names[lbl],
        s=8,
        alpha=0.5,
        edgecolors="none",
    )

ax.set_title("UMAP Projection — Coloured by Sentiment", fontsize=14, fontweight="bold")
ax.set_xlabel("UMAP Dimension 1", fontsize=11)
ax.set_ylabel("UMAP Dimension 2", fontsize=11)
ax.legend(markerscale=3, fontsize=10, title="Sentiment", title_fontsize=11, loc="upper right")
ax.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, "nb04_1_umap_sentiment.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"✅ Plot saved → {plot_path}")
plt.show()


## Section 5 — Cluster Validation

Category Purity, Sentiment Entropy, and Topic Coherence metrics.

In [ ]:
# ── 5.1  Category purity per cluster ───────────────────────────────────────
# Guard: cluster labels must exist.
if "df_merged" not in dir() or "cluster" not in df_merged.columns:
    raise NameError("Run Section 3 first.")

cluster_purity = {}
for cid in sorted(df_merged["cluster"].unique()):
    if cid == -1:
        continue
    cluster_df = df_merged[df_merged["cluster"] == cid]
    top_cats = cluster_df["category"].value_counts(normalize=True).head(3)
    purity = top_cats.sum() * 100
    cluster_purity[cid] = {
        "top_categories": top_cats.to_dict(),
        "purity_pct": purity,
    }
    print(f"Cluster {int(cid):>2.0f}: purity={purity:5.1f}%  |  top 3: {list(top_cats.index)}")

passing = sum(1 for v in cluster_purity.values() if v["purity_pct"] > 40)
print(f"\nClusters with purity > 40%: {passing}/{len(cluster_purity)}")
if passing >= 4:
    print("✅ Category purity target met (≥4 clusters > 40%).")
else:
    print("⚠️  Category purity below target.")


In [ ]:
# ── 5.2  Sentiment entropy per cluster ─────────────────────────────────────
# Shannon entropy: high = mixed sentiment (good); low = sentiment-driven (bad).
cluster_entropy = {}
for cid in sorted(df_merged["cluster"].unique()):
    if cid == -1:
        continue
    cluster_df = df_merged[df_merged["cluster"] == cid]
    dist = cluster_df["label"].value_counts(normalize=True).sort_index()
    # Ensure all 3 labels are represented (fill missing with 0)
    full_dist = pd.Series([dist.get(i, 0) for i in sorted(df_merged["label"].unique())])
    ent = entropy(full_dist, base=2)
    cluster_entropy[cid] = ent
    flag = "⚠️" if ent < 1.0 else "✅"
    print(f"{flag} Cluster {int(cid):>2.0f}: entropy={ent:.3f}")

mean_ent = np.mean(list(cluster_entropy.values()))
print(f"\nMean sentiment entropy: {mean_ent:.3f} (target > 1.0 per cluster)")

flagged_clusters = [cid for cid, ent in cluster_entropy.items() if ent < 1.0]

if flagged_clusters:
    print(f"\n⚠️  {len(flagged_clusters)} cluster(s) with low entropy: {flagged_clusters}")
    if len(flagged_clusters) >= 2:
        print("VALIDATION FAILURE: ≥2 clusters flagged — possible sentiment-driven clustering.")
else:
    print("\n✅ All clusters have high sentiment entropy — not grouping by emotion.")

In [ ]:
# ── 5.3  Combined validation metrics table ─────────────────────────────────
print(f"{'Cluster':>8s} | {'Size':>8s} | {'Purity%':>8s} | {'Entropy':>8s} | {'Top Terms':<40s} | {'Top Categories':<30s} | {'AvgRating':>9s}")
print("-" * 130)

for cid in sorted(df_merged["cluster"].unique()):
    if cid == -1:
        continue
    cluster_df = df_merged[df_merged["cluster"] == cid]
    size = len(cluster_df)
    purity = cluster_purity[cid]["purity_pct"]
    ent = cluster_entropy[cid]
    avg_rating = cluster_df["rating"].mean()

    topic_words = topic_model.get_topic(cid)
    top_terms = ", ".join([w for w, _ in topic_words[:5]]) if topic_words else "(none)"

    top_cats = cluster_purity[cid]["top_categories"]
    top_cat_str = ", ".join([f"{k}({v:.1%})" for k, v in list(top_cats.items())[:3]])

    print(f"{int(cid):8.0f} | {size:8,} | {purity:8.1f} | {ent:8.3f} | {top_terms:<40s} | {top_cat_str:<30s} | {avg_rating:9.2f}")


In [ ]:
# ── 5.4  Sentiment distribution heatmap ────────────────────────────────────
cluster_sentiment = pd.crosstab(
    df_merged[df_merged["cluster"] != -1]["cluster"],
    df_merged[df_merged["cluster"] != -1]["label"],
    normalize="index",
)

# Reorder columns to ensure Neg/Neu/Pos order
cluster_sentiment = cluster_sentiment.reindex(
    columns=sorted(df_merged["label"].unique()), fill_value=0
)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    cluster_sentiment,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    ax=ax,
    cbar_kws={"label": "Proportion"},
)
ax.set_title("Sentiment Distribution per Cluster", fontsize=13, fontweight="bold")
ax.set_xlabel("Sentiment Label", fontsize=11)
ax.set_ylabel("Cluster ID", fontsize=11)

# Color x-axis tick labels by sentiment
for i, label in enumerate(ax.get_xticklabels()):
    label.set_color(SENTIMENT_COLORS[i])

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, "nb04_1_sentiment_heatmap.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"✅ Plot saved → {plot_path}")
plt.show()


In [ ]:
# ── 5.5  Category distribution heatmap (top 10 categories) ─────────────────
top_cats_global = df_merged["category"].value_counts().head(10).index.tolist()

cluster_category = pd.crosstab(
    df_merged[df_merged["cluster"] != -1]["cluster"],
    df_merged[df_merged["cluster"] != -1]["category"],
)
cluster_category = cluster_category[[c for c in top_cats_global if c in cluster_category.columns]]
cluster_category_norm = cluster_category.div(cluster_category.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(
    cluster_category_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    ax=ax,
    cbar_kws={"label": "Proportion"},
)
ax.set_title("Category Distribution per Cluster (Top 10 Categories)", fontsize=13, fontweight="bold")
ax.set_xlabel("Amazon Category", fontsize=11)
ax.set_ylabel("Cluster ID", fontsize=11)
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, "nb04_1_category_heatmap.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"✅ Plot saved → {plot_path}")
plt.show()


## Section 6 — Export Results

JSON exports for HTML/Power BI consumption.

In [ ]:
# ── 6.1  Export cluster assignments ────────────────────────────────────────
if "df_merged" not in dir() or "cluster" not in df_merged.columns:
    raise NameError("Run Section 3 first.")

export_df = df_merged[["text", "cluster", "label", "rating", "category"]].copy()
export_df["review_id"] = export_df.index.astype(int)
export_df["sentiment"] = export_df["label"].astype(int)

# Add UMAP coords from sample (where available)
if "df_sample" in dir() and "umap_x" in df_sample.columns:
    umap_df = df_sample[["umap_x", "umap_y"]].copy()
    export_df = export_df.join(umap_df, how="left")
else:
    export_df["umap_x"] = None
    export_df["umap_y"] = None

records = export_df[["review_id", "text", "cluster", "umap_x", "umap_y", "sentiment", "rating", "category"]].to_dict("records")

assign_path = os.path.join(OUTPUT_DIR, "cluster_assignments.json")
with open(assign_path, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

file_size_mb = os.path.getsize(assign_path) / 1e6
print(f"cluster_assignments.json saved → {assign_path}")
print(f"  Rows : {len(records):,}")
assert len(records) == len(df_merged), f"Row count mismatch: {len(records)} vs {len(df_merged)}"
print(f"  Size : {file_size_mb:.1f} MB")


In [ ]:
# ── 6.2  Export cluster profiles ───────────────────────────────────────────
profiles = []
for cid in sorted(df_merged["cluster"].unique()):
    if cid == -1:
        continue
    cluster_df = df_merged[df_merged["cluster"] == cid]
    topic_words = topic_model.get_topic(cid)
    top_terms = [w for w, _ in topic_words[:10]] if topic_words else []

    top_cats = cluster_df["category"].value_counts(normalize=True).head(3).to_dict()
    purity = cluster_purity[cid]["purity_pct"]
    ent = cluster_entropy[cid]
    avg_rating = cluster_df["rating"].mean()

    profiles.append({
        "cluster_id": int(cid),
        "size": int(len(cluster_df)),
        "pct": round(100 * len(cluster_df) / len(df_merged), 2),
        "top_terms": top_terms,
        "top_categories": top_cats,
        "purity": round(purity, 2),
        "sentiment_entropy": round(ent, 3),
        "avg_rating": round(avg_rating, 2),
    })

profiles_path = os.path.join(OUTPUT_DIR, "cluster_profiles_bge.json")
with open(profiles_path, "w", encoding="utf-8") as f:
    json.dump(profiles, f, ensure_ascii=False, indent=2)

file_size_kb = os.path.getsize(profiles_path) / 1e3
print(f"cluster_profiles_bge.json saved → {profiles_path}")
print(f"  Clusters : {len(profiles)}")
print(f"  Size     : {file_size_kb:.1f} KB")


In [ ]:
# ── 6.3  Export UMAP coordinates ───────────────────────────────────────────
if "df_sample" not in dir() or "umap_x" not in df_sample.columns:
    raise NameError("Run Section 4 first for UMAP coordinates.")

umap_coords = {
    "x": df_sample["umap_x"].tolist(),
    "y": df_sample["umap_y"].tolist(),
    "cluster": df_sample["cluster"].astype(int).tolist(),
    "sentiment": df_sample["label"].astype(int).tolist(),
}

umap_path = os.path.join(OUTPUT_DIR, "umap_coordinates.json")
with open(umap_path, "w", encoding="utf-8") as f:
    json.dump(umap_coords, f, ensure_ascii=False, indent=2)

file_size_mb = os.path.getsize(umap_path) / 1e6
print(f"umap_coordinates.json saved → {umap_path}")
print(f"  Points : {len(umap_coords['x']):,}")
print(f"  Size   : {file_size_mb:.1f} MB")


In [ ]:
# ── 7.0  Results summary + output checklist ────────────────────────────────
print("=" * 60)
print("NOTEBOOK 04.1 — CATEGORY CLUSTERING (BGE + BERTopic)")
print("=" * 60)

print(f"\n📊 Clustering Algorithm : {algorithm_used}")
print(f"   Final n_clusters     : {n_clusters}")
print(f"   Noise points         : {n_noise} ({100*n_noise/len(df_merged):.2f}%)")
if final_min_cluster_size:
    print(f"   HDBSCAN min_cluster_size: {final_min_cluster_size}")

print(f"\n📈 Validation Metrics:")
passing_purity = sum(1 for v in cluster_purity.values() if v['purity_pct'] > 40)
print(f"   Category purity >40% : {passing_purity}/{len(cluster_purity)} clusters")
mean_ent = np.mean(list(cluster_entropy.values()))
print(f"   Mean sentiment entropy: {mean_ent:.3f}")
flagged = [cid for cid, ent in cluster_entropy.items() if ent < 1.0]
print(f"   Clusters flagged (entropy<1.0): {len(flagged)}")

print(f"\n📁 Output Files Checklist:")
files_to_check = [
    ("reviews_filtered.csv", OUTPUT_DIR),
    ("embeddings_bge.npz", MODELS_DIR),
    ("cluster_assignments.json", OUTPUT_DIR),
    ("cluster_profiles_bge.json", OUTPUT_DIR),
    ("umap_coordinates.json", OUTPUT_DIR),
]
all_ok = True
for fname, fdir in files_to_check:
    fpath = os.path.join(fdir, fname)
    ok = os.path.exists(fpath) and os.path.getsize(fpath) > 0
    status = "✅" if ok else "❌"
    print(f"   {status} {fname}")
    if not ok:
        all_ok = False

if all_ok:
    print("\n🎉 All outputs generated successfully.")
else:
    print("\n⚠️  Some outputs are missing.")

print(f"\n➡️  Next Steps:")
print("   1. Notebook 05.1 (Summarization) can read cluster_assignments.json")
print("   2. Web dashboard can load umap_coordinates.json for scatter plots")
print("   3. cluster_profiles_bge.json feeds the cluster overview cards")
